# 01 - Aprendizaje Supervisado

**AI sin humo** - Notas personales para entender deep learning desde cero.

Este es el punto de partida de todo. Acá vemos la idea fundamental detrás del aprendizaje supervisado: qué significa "aprender" para una máquina, cómo se modela un problema, y cómo se entrena un modelo para que haga predicciones útiles. No se necesita saber nada previo.

---

## Contenido

1. [¿Qué es aprender?](#que-es-aprender)
2. [Tipos de aprendizaje](#tipos-de-aprendizaje)
3. [Aprendizaje supervisado en detalle](#aprendizaje-supervisado)
4. [Ejemplo: Regresión lineal 1D](#regresion-lineal-1d)
5. [Función de pérdida (Loss)](#funcion-de-perdida)
6. [El espacio de la loss](#espacio-de-la-loss)
7. [Entrenamiento: gradient descent](#entrenamiento)
8. [Clasificación binaria: regresión logística](#clasificacion-binaria)
9. [Cross-Entropy: la loss para clasificación](#cross-entropy)
10. [Clasificación multiclase: softmax](#clasificacion-multiclase)
11. [¿Por qué usamos log?](#por-que-log)
12. [¿Por qué no usar cross-entropy en regresión?](#por-que-no-ce-regresion)

---

<a id='que-es-aprender'></a>
## 1. ¿Qué es aprender?

Cuando decimos que una máquina "aprende", lo que pasa es bastante simple:

> Tenemos un **modelo** (una función matemática con parámetros) y usamos **datos** provenientes de alguna distribución para **inferir los valores de esos parámetros**, de una forma estadística.

Dicho de otra forma: tenés una ecuación con algunos valores que no conocés, y usás datos reales para ir descubriendo qué valores deberían tener para que la ecuación funcione bien.

**CASI SIEMPRE** que hablamos de aprendizaje, lo que queremos en el fondo es **hacer una predicción** sobre algo:
- Un número (ej: el precio de una casa)
- Una clase binaria (ej: spam o no spam)
- Una clase con muchos posibles resultados (ej: qué letra es esta, qué palabra sigue en un texto)

Eso es todo. No hay magia. Es encontrar una función que prediga bien.

---

<a id='tipos-de-aprendizaje'></a>
## 2. Tipos de aprendizaje

Hay varias formas en las que una máquina puede "aprender":

**Supervisado**: Tenés datos de entrada + respuestas correctas (etiquetas). Le mostrás al modelo muchos ejemplos de "dado esto, la respuesta es esta" y el modelo aprende a mapear inputs → outputs. Es como estudiar con un libro de ejercicios resueltos.

**No supervisado**: Solo tenés datos, sin respuestas. El modelo busca patrones o estructura oculta por su cuenta. Por ejemplo, agrupar clientes similares (clustering) sin decirle qué grupos existen.

**Auto-supervisado**: El modelo genera sus propias "etiquetas" a partir de los datos. Por ejemplo, tapar una palabra en una oración y pedirle que la prediga. Así es como se entrenan los LLMs: les das texto y les pedís que predigan la siguiente palabra.

**Reinforcement Learning (RL)**: Un agente toma acciones en un entorno y recibe recompensas o castigos. Aprende qué acciones maximizan la recompensa a lo largo del tiempo. Así se entrenó a AlphaGo.

En estos notebooks nos enfocamos principalmente en **supervisado** y **auto-supervisado** (que es como se entrenan los LLMs modernos).

---

<a id='aprendizaje-supervisado'></a>
## 3. Aprendizaje supervisado en detalle

La idea del aprendizaje supervisado es tener dos cosas:
- **Datos de entrada** (variables predictoras, features)
- **Target** (lo que queremos predecir, la etiqueta, la respuesta correcta)

Tenemos el concepto de **instancia**, que es un ejemplar de algo. Por ejemplo una casa, con sus características (metros cuadrados, cantidad de habitaciones, barrio, etc.) y una de esas características es la que nos interesa predecir (el precio).

### ¿Qué queremos hacer?

Queremos crear una **FUNCIÓN** (una ecuación matemática) que mapee de los inputs a los outputs. Entonces cuando le mandamos nuevos inputs, nos devuelve un output. A eso le llamamos **inferencia**.

Esa función tiene **parámetros**, y el valor de esos parámetros cambia el resultado de la función. Entonces lo que tenemos es un **tipo de modelo**, que en realidad es una **familia de modelos** finales, porque el modelo final depende de qué valores tengan los parámetros.

Por ejemplo, si nuestro modelo es una recta $y = m \cdot x + b$, la "familia" son todas las rectas posibles. El modelo final depende de qué valores tengan $m$ y $b$.

### ¿Cómo encontramos los mejores parámetros?

Eso es lo que se llama **TRAINING** (entrenamiento) o **learning** (aprendizaje). La idea es que, como ya tenemos pares de inputs → outputs que sabemos que están bien mapeados, la idea es encontrar valores de los parámetros que, dados esos inputs, den los resultados **más parecidos posible** a los outputs que ya conocemos.

### El flujo completo

1. **Elegir una familia de modelos**: una función matemática con parámetros ajustables. Puede ser una recta, un árbol de decisión, una red neuronal, etc.
2. **Definir una función de pérdida (loss)**: mide qué tan mal predice el modelo
3. **Entrenar**: ajustar los parámetros para minimizar la loss, usando los datos que ya tenemos
4. **Inferencia**: usar el modelo entrenado para predecir sobre datos nuevos que nunca vio

### Familias de modelos

Hay muchas familias de funciones aprendibles. Las más conocidas:
- **Modelos lineales**: rectas, planos. Simples pero limitados.
- **Árboles de decisión**: dividen el espacio con preguntas tipo "¿x > 5?". Intuitivos.
- **Gradient boosting**: combinan muchos árboles chiquitos. Muy buenos para datos tabulares.
- **Redes neuronales**: composición de funciones simples. Extremadamente flexibles. Son el foco de estos notebooks.

### Intuición geométrica

Pensalo siempre así: los datos viven en un **espacio multidimensional**, y el target también. Puede ser de una clase o un número, pero siempre vivimos en un espacio donde todo lo que tenemos es **un punto en ese espacio** o **un vector**.

Lo que queremos es encontrar y determinar:
- Qué **zonas** de ese espacio son de una clase o de la otra (clasificación)
- Cuál es la **línea, plano o superficie** que se acercan más a los puntos (regresión)

Cada *modelo* es una función que dibuja una forma (una línea, un plano, una superficie curva) que intenta *separar*, *ajustarse a* o *aproximar* esos puntos.

---

<a id='regresion-lineal-1d'></a>
## 4. Ejemplo: Regresión lineal 1D

Arranquemos con el caso más simple posible para entender todo el flujo.

Tenemos un input $x$ de una dimensión y un output $y$ de una dimensión. O sea, tenemos un espacio de dos dimensiones donde podemos graficar los datos como puntos en un plano.

![Datos de ejemplo](../ai_notas/AI%20notas/image.png)

Vamos a tratar de modelarlo con una familia **LINEAL** de modelos:

$$y = f(x, \phi) = \phi_0 + x \cdot \phi_1$$

Con esta familia podemos armar **cualquier recta**, dependiendo de los valores de los parámetros:
- $\phi_0$: intercepto (el slope, dónde cruza el eje y)
- $\phi_1$: pendiente (cuánto sube o baja por cada unidad de x)

![Distintas rectas para la misma data](../ai_notas/AI%20notas/image%201.png)

Acá se ve claro: **la misma familia de modelos genera distintas rectas** dependiendo de los parámetros. Algunas se ajustan mejor a los datos, otras peor. La pregunta es: **¿cuáles son los mejores parámetros?**

Veámoslo en código:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Sample data: 12 points
x = np.array([0.03, 0.19, 0.34, 0.46, 0.78, 0.81, 1.08, 1.18, 1.39, 1.60, 1.65, 1.90])
y = np.array([0.67, 0.85, 1.05, 1.0, 1.40, 1.5, 1.3, 1.54, 1.55, 1.68, 1.73, 1.6])

In [ ]:
# Our linear model: y = phi0 + phi1 * x
def linear_model(x, phi0, phi1):
    return phi0 + phi1 * x


def plot_model(x, y, phi0, phi1):
    """Plot data and model fit."""
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(x, y, color='black', zorder=5, label='Datos')
    
    x_line = np.linspace(0, 2, 100)
    y_line = linear_model(x_line, phi0, phi1)
    ax.plot(x_line, y_line, 'b-', lw=2, label=f'Modelo: y = {phi0:.1f} + {phi1:.1f}x')
    
    ax.set_xlabel('Input, x')
    ax.set_ylabel('Output, y')
    ax.set_xlim([0, 2.0])
    ax.set_ylim([0, 2.0])
    ax.legend()
    plt.tight_layout()
    plt.show()


# A bad fit: wrong parameters
plot_model(x, y, phi0=0.4, phi1=0.2)

In [ ]:
# A better fit: manually tuned parameters
plot_model(x, y, phi0=0.65, phi1=0.58)

La recta con `phi0=0.4, phi1=0.2` claramente no se ajusta bien a los datos. La de `phi0=0.65, phi1=0.58` se acerca mucho más.

Pero acá los ajustamos **a mano**. Eso no escala. Necesitamos una forma **automática** de encontrar los mejores parámetros. Para eso necesitamos primero definir **qué significa "mejor"**. Ahí entra la función de pérdida.

---

<a id='funcion-de-perdida'></a>
## 5. Función de pérdida (Loss)

¿Cómo hacemos para encontrar la mejor recta? Podemos definir una **FUNCIÓN DE PÉRDIDA** (loss function). Esto es, una función que queramos **OPTIMIZAR** (minimizar en este caso) para acercarnos a la función que nosotros queremos.

La función de pérdida mide **qué tan mal le va al modelo**. Mientras más alta la loss, peor el modelo. Queremos **minimizarla**.

Podemos tener cualquier tipo de función de pérdida. Podría ser la suma o el promedio de las diferencias entre las predicciones y los puntos reales... Pero la más usada para regresión es el **Least Squares Error (LSE)** o **Mean Squared Error (MSE)**:

$$\text{LSE}(\phi) = \sum_{i=1}^{N} (y_i - \hat{y}_i)^2 \quad , \quad \text{MSE}(\phi) = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2$$

### ¿Por qué al cuadrado y no simplemente la diferencia?

Se suele hacer al cuadrado porque tiene **buenas propiedades**:

1. **Hace todo positivo**: si no elevaras al cuadrado, los errores positivos y negativos se compensarían entre sí. Un dato que predecís +5 arriba y otro que predecís -5 abajo daría error total = 0, y eso es mentira.
2. **Penaliza más los errores grandes**: un error de 5 pesa 25, pero uno de 1 pesa solo 1. Esto hace que el modelo se esfuerce más en corregir las predicciones que están más lejos.
3. **Es diferenciable y convexa**: se puede optimizar con gradientes sin problemas. Si usaras valor absoluto (que también es todo positivo), no es diferenciable en el punto 0.

![Visualización de la loss](../ai_notas/AI%20notas/image%202.png)

In [ ]:
def compute_loss(x, y, phi0, phi1):
    """Compute least squares loss."""
    y_hat = linear_model(x, phi0, phi1)
    return np.sum((y - y_hat) ** 2)


# Loss with bad parameters
loss_bad = compute_loss(x, y, phi0=0.4, phi1=0.2)
print(f"Loss (phi0=0.4, phi1=0.2): {loss_bad:.2f}  <-- alta, mal fit")

# Loss with better parameters
loss_good = compute_loss(x, y, phi0=0.65, phi1=0.58)
print(f"Loss (phi0=0.65, phi1=0.58): {loss_good:.2f}  <-- baja, buen fit")

---

<a id='espacio-de-la-loss'></a>
## 6. El espacio de la loss

Ahora tenemos un **nuevo espacio** para pensar: el espacio de la **loss en función de los parámetros**.

Las coordenadas son $\phi_0$ y $\phi_1$ (los parámetros del modelo), y la altura es la loss. Es como un paisaje montañoso donde queremos encontrar el **valle más bajo**.

![Espacio de la loss](../ai_notas/AI%20notas/image%203.png)

Esa es la forma que se genera en la loss. Tenemos una función diferenciable en todos lados, que tenemos que **MINIMIZAR** para encontrar la menor pérdida (que es la diferencia entre los valores reales y lo predicho).

**Esa es la tarea del APRENDIZAJE SUPERVISADO**: encontrar el punto más bajo de esta superficie.

In [ ]:
# Visualize the loss surface
phi0_range = np.linspace(-1, 2, 100)
phi1_range = np.linspace(-1, 2, 100)
PHI0, PHI1 = np.meshgrid(phi0_range, phi1_range)

LOSS = np.zeros_like(PHI0)
for i in range(len(phi0_range)):
    for j in range(len(phi1_range)):
        LOSS[j, i] = compute_loss(x, y, PHI0[j, i], PHI1[j, i])

fig, ax = plt.subplots(figsize=(7, 5))
contour = ax.contourf(PHI0, PHI1, LOSS, levels=50, cmap='viridis')
fig.colorbar(contour, ax=ax, label='Loss')

# Mark the minimum
min_idx = np.unravel_index(LOSS.argmin(), LOSS.shape)
ax.plot(PHI0[min_idx], PHI1[min_idx], 'r*', markersize=15, label='Mínimo')

ax.set_xlabel('$\phi_0$ (intercepto)')
ax.set_ylabel('$\phi_1$ (pendiente)')
ax.set_title('Superficie de la loss: queremos llegar al punto más bajo')
ax.legend()
plt.tight_layout()
plt.show()

---

<a id='entrenamiento'></a>
## 7. Entrenamiento: gradient descent

Ya tenemos:
- Un **modelo** con parámetros
- Una **loss** que mide el error
- Una **superficie de loss** que queremos minimizar

¿Cómo bajamos por esa superficie hasta el punto mínimo? Con un concepto llamado **gradient descent** (descenso por gradiente). El proceso de bajar por esa función hasta el punto mínimo o cercano es lo que se llama **ENTRENAMIENTO**, y para hacerlo usamos los datos que ya tenemos.

![Gradient descent](../ai_notas/AI%20notas/image%204.png)

### ¿Cómo funciona?

Imaginá que estás en una montaña con niebla y querés bajar al valle. No ves nada, pero podés sentir la pendiente del piso bajo tus pies. La estrategia es simple: **caminá siempre en la dirección donde el piso baja más**.

Eso es exactamente gradient descent:

1. **Elegís un punto inicial** (valores random para los parámetros): te parás en algún lugar de la montaña
2. **Calculás la pendiente (gradiente)** en ese punto: sentís para dónde sube el piso
3. **Dás un paso en la dirección contraria** a la pendiente (porque querés **bajar**, no subir)
4. **Repetís** hasta que la loss casi no cambie (llegaste al valle)

La regla de actualización es:

$$\phi \leftarrow \phi - \eta \cdot \nabla L(\phi)$$

Donde:
- $\phi$: parámetros actuales (tu posición)
- $\eta$: **learning rate**, el tamaño del paso. Si es muy grande, te pasás del valle. Si es muy chico, tardás una eternidad.
- $\nabla L(\phi)$: el **gradiente** de la loss, que te dice la dirección y magnitud de la subida

Esto lo vamos a ver **mucho más en detalle** en el notebook de Backpropagation. Por ahora, lo importante es entender la idea: el entrenamiento es un proceso iterativo de ir ajustando los parámetros para reducir el error, guiándose por la pendiente de la loss.

In [ ]:
def gradient_descent(x, y, lr=0.01, n_steps=1000):
    """Simple gradient descent for linear regression."""
    # Random initialization: we start somewhere random in the loss surface
    phi0 = np.random.randn()
    phi1 = np.random.randn()
    n = len(x)
    
    losses = []
    for step in range(n_steps):
        # Forward pass: compute predictions with current params
        y_hat = linear_model(x, phi0, phi1)
        
        # Compute loss (MSE)
        loss = np.mean((y - y_hat) ** 2)
        losses.append(loss)
        
        # Compute gradients: partial derivatives of MSE w.r.t. each parameter
        d_phi0 = -2 / n * np.sum(y - y_hat)       # how loss changes when phi0 changes
        d_phi1 = -2 / n * np.sum((y - y_hat) * x)  # how loss changes when phi1 changes
        
        # Update: take a step in the opposite direction of the gradient
        phi0 -= lr * d_phi0
        phi1 -= lr * d_phi1
    
    return phi0, phi1, losses


phi0_opt, phi1_opt, losses = gradient_descent(x, y, lr=0.1, n_steps=200)
print(f"Parámetros encontrados: phi0={phi0_opt:.3f}, phi1={phi1_opt:.3f}")
print(f"Loss final: {losses[-1]:.4f}")

In [ ]:
# Visualize the training process
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve: should go down over time
axes[0].plot(losses)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Curva de entrenamiento (la loss baja)')

# Final model plotted over data
axes[1].scatter(x, y, color='black', zorder=5, label='Datos')
x_line = np.linspace(0, 2, 100)
y_line = linear_model(x_line, phi0_opt, phi1_opt)
axes[1].plot(x_line, y_line, 'b-', lw=2, label=f'y = {phi0_opt:.2f} + {phi1_opt:.2f}x')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].set_xlim([0, 2.0])
axes[1].set_ylim([0, 2.0])
axes[1].legend()
axes[1].set_title('Modelo entrenado')

plt.tight_layout()
plt.show()

**Resumen hasta acá**: tenemos datos, un modelo lineal, una loss (MSE), y usamos gradient descent para encontrar los parámetros que minimizan la loss. Eso es **aprendizaje supervisado** en su forma más pura.

Ahora vamos a ver qué pasa cuando **no queremos predecir un número** sino **una clase**.

---

<a id='clasificacion-binaria'></a>
## 8. Clasificación binaria: regresión logística

Hasta ahora vimos un problema de **regresión**: predecir un número real.

Ahora vamos con otro tipo de predicción muy común: **decidir una clase**.

Por ejemplo:
- ¿Este mail es *spam* o *no spam*?
- ¿Esta imagen es de un *gato*, *perro* o *caballo*?
- ¿Este usuario va a *comprar* o *no comprar*?

Este tipo de problema se llama **clasificación**. Si hay solo dos clases posibles (como sí o no), se llama **clasificación binaria**.

Seguimos con la misma idea:

> Queremos encontrar una función con parámetros que, dados los datos de entrada, prediga a qué clase pertenece ese punto.

Pero ahora **no nos interesa predecir un número como 37.2**. Queremos **la probabilidad** de que pertenezca a una clase u otra.

### ¿Por qué se llama "regresión logística"?

Se llama así porque sigue teniendo forma parecida a una recta, pero **transformada** para que su resultado esté entre 0 y 1. La transformación que se usa es la **función sigmoide**:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

La sigmoide transforma cualquier número (de $-\infty$ a $+\infty$) en un valor entre 0 y 1, con forma de S.

- Si el resultado es cercano a **1**, el modelo cree que el ejemplo es de la clase positiva
- Si es cercano a **0**, cree que es de la clase negativa
- Si está cerca de **0.5**, no está muy seguro

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


# Visualize sigmoid
z = np.linspace(-6, 6, 200)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(z, sigmoid(z), 'b-', lw=2)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='0.5 (indeciso)')
ax.axhline(y=1.0, color='gray', linestyle=':', alpha=0.3)
ax.axhline(y=0.0, color='gray', linestyle=':', alpha=0.3)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('z (salida de la función lineal)')
ax.set_ylabel('σ(z) (probabilidad)')
ax.set_title('Función Sigmoide: transforma cualquier número a [0, 1]')
ax.set_ylim([-0.05, 1.05])
ax.legend()
plt.tight_layout()
plt.show()

---

<a id='cross-entropy'></a>
## 9. Cross-Entropy: la loss para clasificación

Ahora que tenemos un modelo que predice probabilidades, necesitamos una **loss** adecuada.

**¿Por qué no usar MSE para clasificación?** Porque tiene problemas: los gradientes son poco informativos cuando la predicción está lejos del target. El modelo "no siente" bien la necesidad de corregir.

Ahí entra la función de pérdida más usada en clasificación binaria: **CROSS-ENTROPY** (también llamada **Log Loss**).

Para una muestra es:

$$L = -\left[ y \cdot \log(p) + (1 - y) \cdot \log(1 - p) \right]$$

Donde $p$ es la probabilidad predicha por el modelo y $y$ es la etiqueta real (0 o 1).

### ¿Cómo funciona en la práctica?

**Si la etiqueta real es $y=1$**, la pérdida es $-\log(p)$:
- Si $p = 0.9$ → $-\log(0.9) \approx 0.105$. **Castigo chico**: estuvo casi seguro y acertó.
- Si $p = 0.1$ → $-\log(0.1) \approx 2.302$. **Castigo grande**: estuvo seguro de que era la otra clase y se equivocó feo.

**Si la etiqueta real es $y=0$**, la pérdida es $-\log(1-p)$:
- Si $p = 0.1$ → $-\log(0.9) \approx 0.105$. **Castigo chico**: acertó.
- Si $p = 0.9$ → $-\log(0.1) \approx 2.302$. **Castigo grande**: pifió feo.

### La clave

Este comportamiento **"penaliza más fuerte" las predicciones muy confiadas y equivocadas**. Si el modelo dice "estoy 90% seguro de que es spam" y resulta que no era spam, recibe un castigo enorme. Esto ayuda a que el modelo ajuste sus pesos con gradientes más sólidos cuando más se lo necesita.

![Cross-entropy visual](../ai_notas/AI%20notas/image%205.png)

In [ ]:
def binary_cross_entropy(y_true, p_pred):
    """Binary cross-entropy loss for a single sample."""
    eps = 1e-7  # avoid log(0)
    p_pred = np.clip(p_pred, eps, 1 - eps)
    return -(y_true * np.log(p_pred) + (1 - y_true) * np.log(1 - p_pred))


# Visualize how loss changes with predicted probability
p_range = np.linspace(0.01, 0.99, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# When true label is 1: we want p to be high
axes[0].plot(p_range, binary_cross_entropy(1, p_range), 'b-', lw=2)
axes[0].set_xlabel('Probabilidad predicha (p)')
axes[0].set_ylabel('Loss')
axes[0].set_title('Etiqueta real = 1\n(queremos p alto → loss baja)')

# When true label is 0: we want p to be low
axes[1].plot(p_range, binary_cross_entropy(0, p_range), 'r-', lw=2)
axes[1].set_xlabel('Probabilidad predicha (p)')
axes[1].set_ylabel('Loss')
axes[1].set_title('Etiqueta real = 0\n(queremos p bajo → loss baja)')

plt.tight_layout()
plt.show()

---

<a id='clasificacion-multiclase'></a>
## 10. Clasificación multiclase: softmax

¿Y si tenemos **más de dos clases**? Por ejemplo: tipos de flores, dígitos del 0 al 9, o **tokens en el vocabulario de un LLM** (esto es clave para entender cómo funcionan ChatGPT y similares).

Ahora tenemos $K$ clases posibles.

### El proceso paso a paso

**1. El modelo genera un valor para cada clase**: la salida del modelo genera un valor para cada una de las K clases. Estos valores **no son probabilidades**, son simplemente números que salen de la red. Se llaman **LOGITS**. Hay un logit para cada clase.

**2. Softmax**: después de los logits, lo que se hace es transformarlos en probabilidades. Para eso se usa **softmax**:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

Lo usamos porque esperamos K outputs (que son los parámetros de nuestra distribución categórica) y estos parámetros requieren ser **entre 0 y 1** y **sumar 1** entre todos. Softmax cumple todas esas condiciones.

**3. Loss**: una vez que son probabilidades, solamente agarramos la probabilidad de la **CLASE CORRECTA** y calculamos el negativo del log:

$$L = -\log(p_{\text{clase correcta}})$$

### ¿Por qué se llama cross-entropy?

La fórmula general es:

$$L = -\sum_{i=1}^{K} y_i \cdot \log(p_i)$$

Donde $y$ es un vector **one-hot**: todo ceros excepto un 1 en la posición de la clase correcta.

Si la etiqueta verdadera es la clase $t \in \{1, 2, ..., K\}$, podemos representarla como un vector one-hot. Como estamos haciendo one-hot, donde todo se va a multiplicar por 0 menos 1 solo valor, es lo mismo que hacer: $-\log(p)$ para la clase correcta.

### Diferencias de nombres (son todas lo mismo)

- **Negative Log Likelihood (NLL)**: espera como input las probabilidades ya calculadas (softmax hecho)
- **Cross-Entropy Loss**: espera los logits crudos y computa el softmax adentro de la fórmula
- **Log Loss / BCE**: es para **binario**, espera probabilidades en forma de sigmoid

In [ ]:
def softmax(logits):
    """Compute softmax probabilities from logits."""
    # Subtract max for numerical stability (avoids overflow in exp)
    exp_logits = np.exp(logits - np.max(logits))
    return exp_logits / exp_logits.sum()


def cross_entropy_loss(logits, target_class):
    """Cross-entropy loss for a single sample."""
    probs = softmax(logits)
    return -np.log(probs[target_class])


# Example: 4 classes, model outputs logits
logits = np.array([2.0, 1.0, 0.1, -1.0])
target = 0  # true class is 0

probs = softmax(logits)
loss = cross_entropy_loss(logits, target)

print("Logits (salida cruda del modelo):", logits)
print("Probabilidades (después de softmax):", np.round(probs, 3))
print(f"\nProbabilidad de la clase correcta ({target}): {probs[target]:.3f}")
print(f"Cross-entropy loss: {loss:.3f}")
print(f"\nNota: la loss es baja porque el modelo le da alta probabilidad a la clase correcta.")

In [ ]:
# Compare: what happens when the model is less sure?
logits_unsure = np.array([0.5, 0.4, 0.05, 0.05])  # almost uniform
probs_unsure = softmax(logits_unsure)
loss_unsure = cross_entropy_loss(logits_unsure, target)

print("--- Modelo INSEGURO ---")
print("Probabilidades:", np.round(probs_unsure, 3))
print(f"Loss: {loss_unsure:.3f}  <-- alta")

print("\n--- Modelo SEGURO (y correcto) ---")
print("Probabilidades:", np.round(probs, 3))
print(f"Loss: {loss:.3f}  <-- baja")

print("\nCuando el modelo está más seguro y acierta, la loss es menor.")
print("Eso es lo que queremos: que el modelo aprenda a estar seguro de la respuesta correcta.")

---

<a id='por-que-log'></a>
## 11. ¿Por qué usamos log?

Esta es una pregunta importante. ¿Por qué no usar directamente la probabilidad de la clase correcta como loss?

Cuando estamos entrenando, el modelo no mira de a un ejemplo solo, sino que lo entrenamos de a **batches** (varios ejemplos al mismo tiempo). Queremos maximizar la **probabilidad conjunta** de predecir bien todos los ejemplos. Esto se llama **MAXIMUM LIKELIHOOD**.

Esto implica **MULTIPLICAR** muchas probabilidades:

$$L = \prod_i p_i$$

Y cuando multiplicás muchos números entre 0 y 1, el resultado se vuelve **astronómicamente chico** (underflow numérico). Por ejemplo: $0.9 \times 0.8 \times 0.7 \times ... = 0.00000000...$ muy rápido.

Entonces usamos el truco del **LOG**: como $\log(a \cdot b) = \log(a) + \log(b)$, transformamos multiplicaciones en sumas:

$$\log L = \sum_i \log(p_i)$$

Multiplicación de probs → **suma de log-probs**. Mucho más manejable numéricamente.

Y como queremos **minimizar** (no maximizar), usamos el **negativo** del log-likelihood:

$$L_{\text{NLL}} = -\frac{1}{B} \sum_i \log(p_i)$$

Donde $B$ es el tamaño del batch. Por eso se llama **NEGATIVE LOG LIKELIHOOD**.

El Maximum Likelihood y el Negative Log Likelihood son lo mismo pero en la práctica se usa el log para hacer operaciones con números no tan chicos (que pueden tener errores numéricos importantes).

Además, el log tiene **otras propiedades útiles**:
- Es **diferenciable** y smooth
- Le da **fuerza exponencial** a los errores grandes. En vez de usar $(1-p)$ que es lineal, $-\log(p)$ le da mucho más castigo a los valores cercanos a 0.

---

<a id='por-que-no-ce-regresion'></a>
## 12. ¿Por qué no usar cross-entropy en regresión?

Buena pregunta. Si cross-entropy funciona tan bien para clasificación, ¿por qué no usarla para todo?

Aunque la predicción esté acotada a [0, 1], **si el target es un valor continuo**, cross-entropy no captura bien el error de distancia numérica.

**BCE asume que el target es la probabilidad de una etiqueta binaria** (0 o 1), o como una "etiqueta difusa" en [0, 1] interpretada como probabilidad.

Si el target fuese 0.7, BCE lo trataría como "70% de probabilidad de etiqueta 1" en un Bernoulli, **no como "el número 0.7 exacto"** que queremos predecir.

**Usar BCE con target continuo tiende a forzar la predicción a valores "extremos" (0 o 1)** en vez de acercarla linealmente a 0.7.

### Ejemplo concreto (target = 0.7)

| Predicción | BCE Loss | MSE Loss |
|:----------:|:--------:|:--------:|
| 0.5 | ~0.69 | 0.04 |
| 0.9 | ~1.20 (¡peor!) | 0.04 |

Con BCE, predecir 0.9 da **más pérdida** que predecir 0.5, aunque 0.9 está numéricamente más cerca de 0.7. ¡Eso no tiene sentido para regresión!

Con MSE, ambas predicciones (0.5 y 0.9) equidistan de 0.7 y reciben el mismo castigo. Eso refleja la distancia real.

### Conclusión

- **BCE** (cross-entropy) mide "sorpresa" de eventos binarios o probabilidades de etiquetas, **no** la cercanía de números continuos
- Si el objetivo es un **valor real** → usá **MSE** (o MAE)
- Si el objetivo es una **clase** → usá **cross-entropy**
- Si quisiéramos forzar un modelo probabilístico con datos en [0, 1], habría que usar una distribución continua apropiada (por ejemplo, Beta loss), pero no BCE

---

## Resumen

| Concepto | Descripción |
|----------|-------------|
| **Aprendizaje** | Tener un modelo (función con parámetros) y usar datos para inferir esos parámetros. |
| **Supervisado** | Tenés datos + respuestas correctas. El modelo aprende a mapear inputs → outputs. |
| **Modelo / Familia de modelos** | Una función matemática parametrizada. Los parámetros definen cuál de todos los modelos posibles usamos. |
| **Loss function** | Mide qué tan mal predice el modelo. Se quiere **minimizar**. |
| **MSE** | Para regresión. $(y - \hat{y})^2$. Cuadrado del error. |
| **Cross-entropy** | Para clasificación. $-\log(p)$. Penaliza predicciones confiadas y equivocadas. |
| **Logits** | Valores crudos que salen del modelo, antes de convertirlos en probabilidades. |
| **Softmax** | Transforma logits en probabilidades que suman 1. Para clasificación multiclase. |
| **Sigmoid** | Transforma un número en probabilidad [0, 1]. Para clasificación binaria. |
| **Gradient descent** | Algoritmo iterativo para bajar por la superficie de loss ajustando parámetros. |
| **Learning rate** | Tamaño del paso en gradient descent. Muy grande = inestable, muy chico = lento. |
| **Entrenamiento** | Proceso de encontrar los mejores parámetros usando datos + loss + optimización. |
| **Inferencia** | Usar el modelo entrenado para predecir sobre datos nuevos. |

---

**Siguiente notebook →** [02 - Perceptrón](./02_perceptron.ipynb): la unidad básica que da origen a las redes neuronales.